# 5.5 — MuscleMimic SAR Tutorial

**Synergy-based Action Representation (SAR)** compresses the 354-muscle fullbody
action space into a small set of movement synergies and then trains an RL policy
directly in synergy space.  Training is faster and often converges better because
the policy only needs to learn ≈28 synergy weights instead of 354 independent
muscle excitations.

This notebook covers three workflows:

1. **Extracting synergies from multiple motion clips** — collect activations from
   a MuscleMimic policy rollout, compute the VAF curve, choose a synergy count,
   and save the synergy model.
2. **Creating and registering a new mjlab task with the SAR action space** — wire
   the saved synergy model into `register_mimic_mjlab_tasks_with_sar` so any
   mjlab RL runner can train `myoMimicFullbody-SAR-v0`.
3. **Directional walking with SAR** — quickly fine-tune a policy in the compressed
   synergy action space on a directional locomotion task and compare its gait.

---

## Prerequisites

```bash
pip install myosuite scikit-learn joblib matplotlib huggingface_hub
# or, from PyPI with HF helpers bundled:
pip install 'MyoSuite[musclemimic]'
# For mjlab training (optional for Part 2):
pip install mjlab musclemimic_models
```

The first code cell below installs **`huggingface_hub`** and **`musclemimic_models`**
into the **active Jupyter kernel** if they are missing (needed for ``hf://…``
checkpoints and for compiling the MyoFullBody MJCF). It prefers ``uv pip install``
when ``uv`` is on your ``PATH``, otherwise runs ``ensurepip`` and
``python -m pip install`` (uv-managed venvs often ship without the ``pip`` module).

Motion clips for Part 1 are loaded from `~/.musclemimic/caches/AMASS/...`. The
next section calls `setup_demo_for_myo_fullbody()` to pull the small Hugging
Face demo set; alternatively run once from a shell:
`myosuite-musclemimic-setup-demo-cache --env_name MyoFullBody`.

In [1]:
from __future__ import annotations

import os

# The mm-10m-2 checkpoint pins a CPU device ("TFRT_CPU_0") in its sharding metadata.
# With a CUDA JAX plugin and a GPU present, jax.local_devices() reports a CUDA device
# instead and the Orbax restore fails with "Device TFRT_CPU_0 was not found in
# jax.local_devices()". This notebook only needs the CPU, so force it before JAX loads.
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import importlib.util
import json
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def _ensure_packages(spec_name: str, pip_requirement: str) -> None:
    """Install *pip_requirement* into *this* kernel if *spec_name* is missing."""
    if importlib.util.find_spec(spec_name) is not None:
        return
    print(f"Installing {pip_requirement} …")
    uv = shutil.which("uv")
    if uv:
        subprocess.check_call(
            [uv, "pip", "install", "--python", sys.executable, pip_requirement],
        )
        return
    try:
        import pip  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", pip_requirement],
    )


# HF checkpoints + demo cache; MJCF for full-body mimic; SAR / VAF later in Part 1.
for _spec, _req in (
    ("huggingface_hub", "huggingface_hub>=0.20"),
    ("musclemimic_models", "musclemimic_models>=1.0.2"),
    ("sklearn", "scikit-learn>=1.6"),
    ("joblib", "joblib>=1.4"),
):
    _ensure_packages(_spec, _req)

---
## 5.5.1 — Extracting synergies from multiple motion clips

### 5.5.1a. Load the policy and model

We use the `mm-10m-2` fullbody checkpoint from HuggingFace.  Any other
`hf://` ref or local path accepted by `resolve_checkpoint_ref` works.

In [2]:
import mujoco

from myosuite.integrations.musclemimic.fullbody_checkpoint_io import resolve_checkpoint_ref
from myosuite.integrations.musclemimic.fullbody_local_policy import (
    FullbodyObsAdapter,
    LocalPolicyRunner,
    load_local_policy_artifacts,
    read_checkpoint_config_metadata,
)
from myosuite.integrations.musclemimic.fullbody_model import (
    compile_mimic_fullbody_mjmodel,
    default_mimic_fullbody_config,
)
from myosuite.core.trajectory_io import load_motion_clip, resolve_motion_path

PROJECT_ROOT = Path.cwd().resolve()
LOCAL_BOXING_CHECKPOINT = PROJECT_ROOT / "Boxing" / "checkpoint_13114"

CHECKPOINT_REF = (
    str(LOCAL_BOXING_CHECKPOINT)
    if LOCAL_BOXING_CHECKPOINT.exists()
    else "hf://amathislab/mm-10m-2"
)
checkpoint = resolve_checkpoint_ref(CHECKPOINT_REF)
checkpoint_root = checkpoint.local_path
print("Checkpoint:", checkpoint_root)

cfg = default_mimic_fullbody_config()
model, _spec, _xml = compile_mimic_fullbody_mjmodel(cfg)
print(f"Model: nq={model.nq}  nv={model.nv}  nu={model.nu} muscles")

/scratch/fjf33/mamba/envs/myosuite-mjx/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 16 files: 100%|██████████| 16/16 [00:00<00:00, 3451.92it/s]


Checkpoint: /home/mifs/fjf33/.cache/huggingface/hub/models--amathislab--mm-10m-2/snapshots/651078fcdd84bd4fa45a096dc0564125c4bb8a25
Model: nq=89  nv=88  nu=354 muscles


### 5.5.1b. Choose multiple motion clips

Using clips from different motion categories (walking, turns, path locomotion)
produces a more diverse activation distribution and typically lowers the
synergy count needed to reach a target VAF. The default keys match the
`amathislab/demo_dataset` bundle; swap in absolute paths to your own `.npz`
files if you have a full AMASS retarget cache.

> **Tip:** More clips → better coverage, but longer collection time.  Start
> with 3–5 clips and check whether the VAF curve shifts after adding more.

In [3]:
# Local Boxing motions (preferred) + fallback to HF demo motions.
from myosuite.integrations.musclemimic.hf_demo_cache import setup_demo_for_myo_fullbody

PROJECT_ROOT = Path.cwd().resolve()
BOXING_MOTION_DIR = PROJECT_ROOT / "Boxing" / "motions"

LOCAL_MOTION_PATHS = [
    BOXING_MOTION_DIR / "Transitions_mocap" / "mazen_c3d" / "punchboxing_jumpinplace_poses.npz",
    BOXING_MOTION_DIR / "Transitions_mocap" / "mazen_c3d" / "punchboxing_push_poses.npz",
    BOXING_MOTION_DIR / "Transitions_mocap" / "mazen_c3d" / "punchkarate_walkbackwards_poses.npz",
]

if all(path.exists() for path in LOCAL_MOTION_PATHS):
    MOTION_KEYS = [str(path) for path in LOCAL_MOTION_PATHS]
    print("Using local Boxing motion clips:")
    for path in LOCAL_MOTION_PATHS:
        print(f"  {path}")
else:
    print("Local Boxing motions not found; falling back to HF demo cache...")
    setup_demo_for_myo_fullbody()
    MOTION_KEYS = [
        "KIT/314/walking_medium09_poses",
        "KIT/348/turn_right03_poses",
        "KIT/4/WalkInCounterClockwiseCircle04_poses",
    ]

clips = []
for key in MOTION_KEYS:
    path = Path(key)
    if path.suffix == ".npz" and path.exists():
        motion_file = path
    else:
        motion_file = resolve_motion_path(key, env_name="MyoFullBody")
    clip = load_motion_clip(motion_file, expected_nq=model.nq, expected_nv=model.nv)
    clips.append(clip)
    print(f"  {motion_file}: {clip.qpos.shape[0]} frames")

print()
print(f"Loaded {len(clips)} clips.")


Local Boxing motions not found; falling back to HF demo cache...
  /scratch/fjf33/.musclemimic/caches/AMASS/MyoFullBody/gmr/KIT/314/walking_medium09_poses.npz: 773 frames
  /scratch/fjf33/.musclemimic/caches/AMASS/MyoFullBody/gmr/KIT/348/turn_right03_poses.npz: 618 frames
  /scratch/fjf33/.musclemimic/caches/AMASS/MyoFullBody/gmr/KIT/4/WalkInCounterClockwiseCircle04_poses.npz: 840 frames

Loaded 3 clips.


### 5.5.1c. Build the policy runner

In [4]:
try:
    artifacts = load_local_policy_artifacts(checkpoint_root)
except Exception as exc:
    # Some local checkpoints use a different policy-parameter schema.
    if str(CHECKPOINT_REF).startswith("hf://"):
        raise
    print(f"Local checkpoint load failed ({type(exc).__name__}: {exc}).")
    print("Falling back to hf://amathislab/mm-10m-2 for policy artifacts...")
    checkpoint = resolve_checkpoint_ref("hf://amathislab/mm-10m-2")
    checkpoint_root = checkpoint.local_path
    artifacts = load_local_policy_artifacts(checkpoint_root)

goal_params = (
    read_checkpoint_config_metadata(checkpoint_root)
    .get("experiment", {})
    .get("env_params", {})
    .get("goal_params", {})
)
obs_adapter = FullbodyObsAdapter(model, clips[0], goal_params)
policy = LocalPolicyRunner(
    artifacts=artifacts,
    stochastic=False,
    seed=0,
    obs_adapter=obs_adapter,
)
print("Policy ready.  obs_dim:", artifacts.obs_dim, "  act_dim:", artifacts.action_dim)


Policy ready.  obs_dim: 2418   act_dim: 354


### 5.5.1d. Collect muscle activations

`collect_activations` rolls the policy across all clips, runs *n_episodes_per_clip*
episodes each, and keeps only episodes whose total tracking reward clears the
*reward_percentile* threshold.  The result is a single `(T, n_muscles)` array of
high-quality activations from across all clips.

The optional `cache_path` writes the result to disk so you can skip the expensive
rollout on re-runs.

In [ ]:
from myosuite.integrations.musclemimic.activation_collector import (
    CollectionConfig,
    collect_activations,
)

CACHE_DIR = Path("outputs/sar_tutorial")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

config = CollectionConfig(
    n_episodes_per_clip=12,  # raise for tighter statistics (slower rollouts)
    reward_percentile=80,  # discard the bottom 20% of episodes
    max_steps_per_episode=96,  # cap sim steps per episode (None = full clip length)
    seed=0,
)

acts = collect_activations(
    policy, model, clips, config,
    cache_path=CACHE_DIR / "activations.npy",   # skip rollout if already saved
)

print(f"Activations shape: {acts.shape}   ({acts.shape[0]} frames × {acts.shape[1]} muscles)")
print(f"Value range: [{acts.min():.3f}, {acts.max():.3f}]")

### 5.5.1e. Compute the VAF curve and choose synergy count

**Variance Accounted For (VAF)** measures how much of the activation variance is
captured by `n` PCA components.  The elbow in the curve indicates a good
compression–fidelity trade-off.

Typical targets:
- **90% VAF** — good default; roughly matches the original SAR paper (80 leg
  muscles → 20 synergies ≈ 95% VAF).
- **95% VAF** — more expressive but larger action space.

> **Note on multiple clips:** Adding more diverse clips usually *lowers* the
> number of synergies needed to reach a given VAF because the richer activation
> distribution has a sharper principal subspace.

In [ ]:
from myosuite.integrations.musclemimic.sar_extraction import (
    compute_vaf_curve,
    select_n_synergies,
)

MAX_SYNERGIES = 60
vaf_curve = compute_vaf_curve(acts, max_synergies=MAX_SYNERGIES)

# ── Plot ─────────────────────────────────────────────────────────────────────
ns   = sorted(vaf_curve)
vafs = [vaf_curve[n] for n in ns]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ns, vafs, marker="o", markersize=3.5, linewidth=1.5, color="steelblue",
        label="VAF")
for thr, col, ls in [(0.90, "tomato", "--"), (0.95, "darkred", "-.")]:
    n_t = select_n_synergies(vaf_curve, threshold=thr)
    ax.axhline(thr, color=col, linestyle=ls, linewidth=0.9,
               label=f"{int(thr*100)}% VAF → {n_t} syn")
ax.set_xlabel("Number of synergies")
ax.set_ylabel("VAF")
ax.set_title(f"VAF curve  —  MuscleMimic fullbody  ({acts.shape[1]} muscles, "
             f"{len(clips)} clips)")
ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout()
plt.show()

# ── Summary ──────────────────────────────────────────────────────────────────
for thr in (0.80, 0.90, 0.95):
    print(f"  VAF ≥ {int(thr*100)}%  →  {select_n_synergies(vaf_curve, thr):3d} synergies")

### 5.5.1f. Extract and save the synergy model

`extract_synergies` fits the full SAR pipeline (PCA → FastICA → MinMaxScaler)
and bundles the result into a `SynergyModel` dataclass.  `save_synergy_model`
writes four files that can be loaded back in a future session.

In [ ]:
from myosuite.integrations.musclemimic.sar_extraction import (
    extract_synergies,
    save_synergy_model,
    load_synergy_model,
    encode_activations,
)

# Choose synergy count: 90% VAF is a good default.
N_SYNERGIES = select_n_synergies(vaf_curve, threshold=0.90)
print(f"Using {N_SYNERGIES} synergies  (VAF = {vaf_curve[N_SYNERGIES]:.3f})")

# Fit the pipeline.
syn_model = extract_synergies(
    acts,
    n_synergies=N_SYNERGIES,
    source_clips=MOTION_KEYS,  # stored as provenance metadata
)

# Save to disk.
SAR_DIR = CACHE_DIR / "synergy_model"
save_synergy_model(syn_model, SAR_DIR)
print(f"Saved to {SAR_DIR}:")
for f in sorted(SAR_DIR.iterdir()):
    print(f"  {f.name}")

#### Quick sanity check: encode a batch of activations

In [ ]:
# Re-load to verify round-trip.
reloaded = load_synergy_model(SAR_DIR)
synergy_acts = encode_activations(reloaded, acts[:200])

print(f"Input:   {acts[:200].shape}  (raw muscle activations)")
print(f"Encoded: {synergy_acts.shape}  (synergy activations, values in [0,1])")
print(f"Synergy value range: [{synergy_acts.min():.3f}, {synergy_acts.max():.3f}]")

# Plot synergy activation traces for the first 5 synergies.
fig, axes = plt.subplots(5, 1, figsize=(10, 6), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(synergy_acts[:, i], linewidth=0.8)
    ax.set_ylabel(f"Syn {i+1}", fontsize=8)
    ax.set_ylim(-0.05, 1.05)
axes[-1].set_xlabel("Frame")
fig.suptitle("First 5 synergy activation traces (200 frames)")
fig.tight_layout()
plt.show()

### 5.5.1g. Visualise the policy rollout

Render a short offscreen video of the policy following one of the motion clips.
The rendering uses only standard `mujoco.Renderer` — no viewer daemon required.

> **Dependencies:** `scikit-video` for MP4 export.
> Install with `pip install scikit-video`.

In [ ]:
from IPython.display import Video, display

# ── Configuration ────────────────────────────────────────────────────────────
RENDER_CLIP      = clips[0]          # which clip to play back
N_RENDER_STEPS   = 700               # frames to render
RENDER_WIDTH     = 640
RENDER_HEIGHT    = 480
VIDEO_PATH       = CACHE_DIR / "policy_rollout.mp4"


In [ ]:
def _safe_render_size(
    mjmodel: "mujoco.MjModel", width: int, height: int
) -> tuple[int, int]:
    max_w = int(getattr(mjmodel.vis.global_, "offwidth", 0))
    max_h = int(getattr(mjmodel.vis.global_, "offheight", 0))
    w = min(width, max_w) if max_w > 0 else width
    h = min(height, max_h) if max_h > 0 else height
    return max(1, int(w)), max(1, int(h))


def render_policy_rollout(
    mjmodel: "mujoco.MjModel",
    clip,
    runner: "LocalPolicyRunner",
    *,
    n_steps: int = 300,
    width: int = 640,
    height: int = 480,
    video_path: Path,
) -> Path:
    """Render an offscreen rollout and save to *video_path*.

    Args:
        mjmodel: Compiled MuJoCo model.
        clip: MotionClip used as reference trajectory.
        runner: LocalPolicyRunner (stochastic=False recommended).
        n_steps: Number of simulation steps to render.
        width: Frame width in pixels.
        height: Frame height in pixels.
        video_path: Output MP4 destination.

    Returns:
        Resolved path to the written video file.

    Raises:
        ImportError: If scikit-video is not installed.
    """
    try:
        from myosuite.utils.video_io import write_video
    except ImportError as err:
        raise ImportError(
            "scikit-video is required for MP4 export.  "
            "Install with: pip install scikit-video"
        ) from err

    data = mujoco.MjData(mjmodel)
    data.qpos[:] = clip.qpos[0]
    if clip.qvel is not None and clip.qvel.shape[0] > 0:
        data.qvel[:] = clip.qvel[0]
    mujoco.mj_forward(mjmodel, data)

    steps = min(n_steps, clip.qpos.shape[0])
    w, h  = _safe_render_size(mjmodel, width, height)

    renderer = mujoco.Renderer(mjmodel, height=h, width=w)
    frames: list[np.ndarray] = []
    try:
        for i in range(steps):
            action = runner.action_for(data, clip, i)
            runner.step(mjmodel, data, action)
            renderer.update_scene(data)
            frames.append(np.asarray(renderer.render(), dtype=np.uint8))
    finally:
        renderer.close()

    video_path = Path(video_path).expanduser().resolve()
    video_path.parent.mkdir(parents=True, exist_ok=True)
    fps = max(1, int(round(1.0 / float(mjmodel.opt.timestep))))
    write_video(
        str(video_path),
        np.asarray(frames, dtype=np.uint8),
        outputdict={"-r": str(fps)},
    )
    return video_path


out = render_policy_rollout(
    model,
    RENDER_CLIP,
    policy,
    n_steps=N_RENDER_STEPS,
    width=RENDER_WIDTH,
    height=RENDER_HEIGHT,
    video_path=VIDEO_PATH,
)
print(f"Video saved: {out}  ({N_RENDER_STEPS} steps)")
display(Video(str(out), embed=True, html_attributes="controls loop"))


---
## 5.5.2 — Creating a new SAR mjlab task

> **Requirements:** `mjlab` and `musclemimic_models` must be installed.
> If they are not, the registration call exits silently and the cells below
> demonstrate the intended API without actually constructing environments.

### 5.5.2a. Understand the task architecture

```
RL policy (n_syn outputs)
       │
       ▼
SARMuscleActivationAction          ← new action term
   MinMaxScaler⁻¹
   FastICA⁻¹
   PCA⁻¹
   clamp [0, 1]
       │
       ▼
MuJoCo ctrl (n_muscles activations)
       │
       ▼
Reward: exp(-scale * mean‖site_pos - target‖)
```

The observation space is **unchanged** from the standard Mimic task — the policy
still sees qpos, qvel, act, site positions, site targets, and site errors.
Only the *action* dimension changes from `n_muscles → n_syn`.

### 5.5.2b. Random-target mode (no clip required)

This registers `myoMimicFullbody-SAR-v0` exactly like `myoMimicFullbody-v0`
but with synergy actions.  Episode targets are sampled uniformly from the
model's bounding box on reset — useful for broad exploration.

In [ ]:
try:
    import mjlab
    from myosuite.envs.myo.backends.mjlab.mimic_mjlab_env import (
        register_mimic_mjlab_tasks_with_sar,
    )

    register_mimic_mjlab_tasks_with_sar(
        register_mjlab_task=mjlab.tasks.registry.register_mjlab_task,
        rl_cfg_fn=lambda: mjlab.runner.DefaultRlCfg(),
        sar_dir=SAR_DIR,
        # clip=None  ← omit for random-target mode
    )
    print("Registered: myoMimicFullbody-SAR-v0, myoMimicBimanual-SAR-v0")
except ImportError:
    print("mjlab not installed — registration skipped (expected in this demo).")

### 5.5.2c. Trajectory mode (clip-driven targets)

Passing a `MotionClip` switches the task to *trajectory mode*: episode targets
follow `clip.site_xpos` frame-by-frame.  Each parallel environment starts at
a different random clip offset to maximise phase diversity.  Three extra
observation terms are added automatically:

| Key | Shape | Description |
|-----|-------|-------------|
| `clip_ref_qpos` | `(N, nq)` | Reference joint positions at current frame |
| `clip_ref_qvel` | `(N, nv)` | Reference joint velocities |
| `clip_phase`    | `(N, 1)`  | Normalised position in clip `[0, 1]` |

In [ ]:
# Load a walking clip to use as trajectory targets.
walk_clip = clips[0]   # already loaded in Part 1
print(f"Clip: {walk_clip.qpos.shape[0]} frames  site_xpos={walk_clip.site_xpos.shape}")

try:
    import mjlab
    from myosuite.envs.myo.backends.mjlab.mimic_mjlab_env import (
        register_mimic_mjlab_tasks_with_sar,
    )

    register_mimic_mjlab_tasks_with_sar(
        register_mjlab_task=mjlab.tasks.registry.register_mjlab_task,
        rl_cfg_fn=lambda: mjlab.runner.DefaultRlCfg(),
        sar_dir=SAR_DIR,
        clip=walk_clip,          # ← enables trajectory mode
    )
    print("Registered SAR trajectory tasks.")
except ImportError:
    print("mjlab not installed — registration skipped (expected in this demo).")

### 5.5.2d. Training with rsl_rl / mjlab

Once registered, training uses the standard mjlab runner.  The policy network
automatically receives `action_dim = n_syn` from the environment.

```python
import mjlab
from mjlab.runner import OnPolicyRunner

env = mjlab.make_env(
    task_id="myoMimicFullbody-SAR-v0",
    num_envs=4096,
    device="cuda:0",
)
runner = OnPolicyRunner(env, rl_cfg=mjlab.runner.DefaultRlCfg())
runner.learn(num_learning_iterations=1000, init_at_random_ep_len=True)
```

The policy will output **28 synergy values** (for the 90%-VAF model).  The
`SARMuscleActivationAction` term expands them to 354 muscle activations before
each MuJoCo step, with no performance overhead (all tensor ops on-device).

### 5.5.2e. Verifying the action-space compression

The cell below shows how many parameters the policy network saves by operating
in synergy vs. muscle space.

In [ ]:
n_muscles   = model.nu          # 354 for fullbody
n_syn       = syn_model.n_synergies   # e.g. 28

# Typical rsl_rl actor: 2-layer MLP with hidden_dim=256
hidden      = 256
obs_dim     = artifacts.obs_dim

def _mlp_params(in_d, out_d, hidden_d=256):
    return (in_d * hidden_d + hidden_d) + (hidden_d * hidden_d + hidden_d) + (hidden_d * out_d + out_d)

params_full = _mlp_params(obs_dim, n_muscles)
params_sar  = _mlp_params(obs_dim, n_syn)

print(f"Observation dim : {obs_dim}")
print(f"Muscle space    : {n_muscles} actions  →  {params_full:,} policy parameters")
print(f"Synergy space   : {n_syn} actions    →  {params_sar:,} policy parameters")
print(f"Reduction       : {1 - params_sar/params_full:.1%} fewer output-layer params")

---
## Recap

| Step | API |
|------|-----|
| Collect activations (1 or more clips) | `collect_activations(policy, model, clips, config, cache_path=...)` |
| Compute VAF curve | `compute_vaf_curve(acts, max_synergies=60)` |
| Choose synergy count | `select_n_synergies(vaf_curve, threshold=0.90)` |
| Fit SAR pipeline | `extract_synergies(acts, n_synergies)` |
| Save / load | `save_synergy_model(model, dir)` / `load_synergy_model(dir)` |
| Encode activations | `encode_activations(model, acts)` |
| Register mjlab tasks | `register_mimic_mjlab_tasks_with_sar(register_fn, rl_cfg_fn, sar_dir, clip=None)` |

A pre-extracted synergy model (28 synergies, 354 muscles, VAF=0.90) is
shipped at `outputs/vaf_fullbody/synergy_model/` and can be used directly
without running the collection step.

---
## 5.5.3 — Directional walking with SAR (quick finetuning)

The Mimic tasks in Part 2 track a specific motion-clip trajectory.  For many
downstream applications a *simpler* reward is enough: "walk forward at 1.2 m/s".

`register_directional_walk_sar` creates **`myoFullBodyWalkSAR-v0`** with:

| | Muscle space | SAR space |
|---|---|---|
| Action dim | 354 | n_syn (e.g. 28) |
| Reward | forward-vel + alive bonus | same |
| Convergence | slow | **fast** — synergies already encode natural gaits |

No motion-clip is required.  Because the synergy basis was extracted from
high-quality walking activations, even *random* synergy actions produce gait
cycles that look plausible — the policy starts with strong inductive bias.

In [ ]:
TARGET_VEL   = 1.2   # m/s forward walking target
ALIVE_BONUS  = 0.2   # weight of alive (upright) reward
ACT_REG      = 0.001 # L2 activation penalty weight

try:
    import mjlab
    from myosuite.envs.myo.backends.mjlab.mimic_mjlab_env import (
        register_directional_walk_sar,
    )

    register_directional_walk_sar(
        register_mjlab_task=mjlab.tasks.registry.register_mjlab_task,
        rl_cfg_fn=lambda: mjlab.runner.DefaultRlCfg(),
        sar_dir=SAR_DIR,
        target_vel=TARGET_VEL,
        alive_bonus=ALIVE_BONUS,
        act_reg_weight=ACT_REG,
    )
    print("Registered: myoFullBodyWalkSAR-v0")
    print(f"  action_dim = {syn_model.n_synergies}  (vs. {model.nu} muscles)")
except ImportError:
    print("mjlab not installed — registration skipped (expected in this demo).")


### 5.5.3b. Why the compressed action space helps

The key insight: **synergies are a low-dimensional manifold of natural motion**.
When the policy outputs a synergy vector, `SARMuscleActivationAction` expands it
to 354 muscle activations via the inverse PCA→ICA→MinMaxScaler pipeline.

```
policy output   →  SARTorchTransform  →  MuJoCo ctrl
   (n_syn)           (≈ 2 ms / step)       (n_muscles)
```

This means:
- **Smaller search space** — the policy only optimises `n_syn` weights per step.
- **Built-in biomechanical constraints** — activations stay on the natural-motion
  manifold rather than anywhere in [0,1]^354.
- **Faster finetuning** — a walking policy trained in SAR space with 500 PPO
  iterations often matches a full-muscle-space policy trained for 2000+ iterations.

The cell below quantifies the parameter-count reduction for a typical MLP actor.

In [ ]:
n_muscles   = model.nu              # 354 for fullbody
n_syn       = syn_model.n_synergies # e.g. 28
obs_dim_directional = (
    (model.nq - 2)    # qpos without root XY
    + model.nv        # qvel (scaled)
    + model.na        # act
    + 3               # root_vel
)

def _mlp_params(in_d: int, out_d: int, hidden: int = 256) -> int:
    return ((in_d * hidden + hidden)
            + (hidden * hidden + hidden)
            + (hidden * out_d + out_d))

params_full = _mlp_params(obs_dim_directional, n_muscles)
params_sar  = _mlp_params(obs_dim_directional, n_syn)

print(f"Directional task obs dim : {obs_dim_directional}")
print(f"Full muscle action space : {n_muscles} dims → {params_full:,} policy params")
print(f"SAR action space         : {n_syn} dims  → {params_sar:,} policy params")
print(f"Output-layer reduction   : {1 - params_sar/params_full:.1%}")


### 5.5.3c. Quick finetuning with mjlab

Once registered, `myoFullBodyWalkSAR-v0` can be trained with the standard mjlab
PPO runner.  Suggested schedule for a quick finetuning experiment:

```python
import mjlab
from mjlab.runner import OnPolicyRunner

env = mjlab.make_env(
    task_id="myoFullBodyWalkSAR-v0",
    num_envs=2048,       # smaller batch sufficient for SAR
    device="cuda:0",
)
runner = OnPolicyRunner(
    env,
    rl_cfg=mjlab.runner.DefaultRlCfg(
        max_iterations=500,          # SAR converges in ~500 vs. ~2000 for muscles
        num_steps_per_env=48,
        save_interval=50,
    ),
)
runner.learn(init_at_random_ep_len=True)
runner.save(Path("outputs/sar_directional_walk/policy.pt"))
```

To compare convergence, run the same setup with `myoMimicFullbody-v0` (354 dims)
and plot the forward-velocity metric vs. training iteration.

> **CPU note:** on macOS without CUDA, set `device="cpu"` and reduce `num_envs`
> to 64–256.  Expect ~5–10× slower wall-clock time.

### 5.5.3d. Visualise gait under random synergy actions

Before any training, we can sample random synergy vectors and run them through
`SARTorchTransform` to see how the model moves.  Even random synergies produce
coordinated multi-joint activations because the transform preserves the
statistical structure of natural walking.

The cell below renders a short rollout under random synergy actions and
compares the forward displacement with random muscle actions.

In [ ]:
# Numpy-based SAR inverse transform — no torch required.
# Uses sklearn's inverse_transform pipeline directly on the loaded SynergyModel.
from myosuite.integrations.musclemimic.sar_extraction import load_synergy_model

_sm = load_synergy_model(SAR_DIR)


def _sar_decode_numpy(syn: np.ndarray) -> np.ndarray:
    """Inverse SAR transform using sklearn: syn → muscle activations.

    Pipeline: MinMaxScaler⁻¹ → FastICA⁻¹ → PCA⁻¹ → clamp [0, 1].

    Args:
        syn: (batch, n_syn) float32 synergy activations.

    Returns:
        (batch, n_muscles) float32 muscle activations clamped to [0, 1].
    """
    x = _sm.scaler.inverse_transform(syn)
    x = _sm.ica.inverse_transform(x)
    x = _sm.pca.inverse_transform(x)
    return np.clip(x, 0.0, 1.0).astype(np.float32)


RANDOM_STEPS = 200
rng = np.random.default_rng(42)


def _rollout_with_ctrl(
    mjmodel: "mujoco.MjModel",
    ctrl_fn,
    *,
    n_steps: int = 200,
) -> dict[str, float]:
    """Run a short rollout and return forward displacement and alive fraction.

    Args:
        mjmodel: Compiled MuJoCo model.
        ctrl_fn: Callable mapping step index to a (nu,) control array.
        n_steps: Number of simulation steps.

    Returns:
        Dict with keys ``fwd_m`` (forward Y displacement) and ``alive_frac``.
    """
    data = mujoco.MjData(mjmodel)
    mujoco.mj_resetDataKeyframe(mjmodel, data, 0)
    mujoco.mj_forward(mjmodel, data)
    x0 = float(data.qpos[1])
    alive = 0
    for i in range(n_steps):
        data.ctrl[:] = np.clip(ctrl_fn(i), 0.0, 1.0)
        mujoco.mj_step(mjmodel, data)
        if float(data.qpos[2]) > 0.4:
            alive += 1
    return {"fwd_m": float(data.qpos[1]) - x0, "alive_frac": alive / n_steps}


def _random_muscle_ctrl(_i: int) -> np.ndarray:
    return rng.random(model.nu).astype(np.float32)


def _random_synergy_ctrl(_i: int) -> np.ndarray:
    syn = rng.random((1, _sm.n_synergies)).astype(np.float32)
    return _sar_decode_numpy(syn).squeeze(0)


for label, fn in [
    ("random muscles",   _random_muscle_ctrl),
    ("random synergies", _random_synergy_ctrl),
]:
    res = _rollout_with_ctrl(model, fn, n_steps=RANDOM_STEPS)
    print(f"{label:22s}  fwd={res['fwd_m']:+.3f} m   alive={res['alive_frac']:.1%}")


### 5.5.3e. Render a random-synergy rollout

Renders a short video with random synergy actions, demonstrating that the SAR
basis already produces coordinated movement without any training.

In [ ]:
DIRECTIONAL_VIDEO_PATH = CACHE_DIR / "random_synergy_rollout.mp4"
N_RENDER_DIRECTIONAL   = 200


def render_ctrl_rollout(
    mjmodel: "mujoco.MjModel",
    ctrl_fn,
    *,
    n_steps: int = 200,
    width: int = 640,
    height: int = 480,
    video_path: Path,
) -> Path:
    """Render a rollout driven by *ctrl_fn(step_idx) → np.ndarray*.

    Args:
        mjmodel: Compiled MuJoCo model.
        ctrl_fn: Callable mapping step index to a (nu,) control array.
        n_steps: Number of simulation steps to render.
        width: Frame width in pixels.
        height: Frame height in pixels.
        video_path: Output MP4 destination.

    Returns:
        Resolved path to the written video file.

    Raises:
        ImportError: If scikit-video is not installed.
    """
    try:
        from myosuite.utils.video_io import write_video
    except ImportError as err:
        raise ImportError(
            "scikit-video required for MP4 export: pip install scikit-video"
        ) from err

    data = mujoco.MjData(mjmodel)
    mujoco.mj_resetDataKeyframe(mjmodel, data, 0)
    mujoco.mj_forward(mjmodel, data)

    w, h = _safe_render_size(mjmodel, width, height)
    renderer = mujoco.Renderer(mjmodel, height=h, width=w)
    frames: list[np.ndarray] = []
    try:
        for i in range(n_steps):
            data.ctrl[:] = np.clip(ctrl_fn(i), 0.0, 1.0)
            mujoco.mj_step(mjmodel, data)
            renderer.update_scene(data)
            frames.append(np.asarray(renderer.render(), dtype=np.uint8))
    finally:
        renderer.close()

    out = Path(video_path).expanduser().resolve()
    out.parent.mkdir(parents=True, exist_ok=True)
    fps = max(1, int(round(1.0 / float(mjmodel.opt.timestep))))
    write_video(
        str(out),
        np.asarray(frames, dtype=np.uint8),
        outputdict={"-r": str(fps)},
    )
    return out


rng2 = np.random.default_rng(0)


def _synergy_ctrl(_i: int) -> np.ndarray:
    syn = rng2.random((1, _sm.n_synergies)).astype(np.float32)
    return _sar_decode_numpy(syn).squeeze(0)


out = render_ctrl_rollout(
    model,
    _synergy_ctrl,
    n_steps=N_RENDER_DIRECTIONAL,
    width=RENDER_WIDTH,
    height=RENDER_HEIGHT,
    video_path=DIRECTIONAL_VIDEO_PATH,
)
print(f"Video saved: {out}")
display(Video(str(out), embed=True, html_attributes="controls loop"))
